# Tool Notebook
This notebook implements the core tool-function scaffolding for SS03 and prepares reusable schemas for later tool-calling lessons.


In [45]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## 1. Initialize SDK Client
This cell loads environment variables and creates the Anthropic client and model identifier used by helper utilities in subsequent cells.

[Review SS03 Notes](../s06_tool_use_with_claude/ss03_tool_functions/index.md)

In [46]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## 2. Build Conversation Helpers
These helper functions keep message construction consistent and define a `chat` wrapper that forwards standardized parameters to the Messages API.

[Review SS03 Notes](../s06_tool_use_with_claude/ss03_tool_functions/index.md)
[Preview SS05 Notes: Multi-Block Message Handling](../s06_tool_use_with_claude/ss05_handling_message_blocks/index.md)

In [47]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

## 4. Add a Typed Tool Schema
This section introduces ToolParam and defines get_current_datetime with a strict JSON schema so Claude can reliably understand valid arguments and output formatting rules.

[Open SS04 Lesson Notes: Tool Schemas](../s06_tool_use_with_claude/ss04_tool_schemas/index.md)

In [48]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("data_format cannot be empty")
    return datetime.now().strftime(date_format)

# get_current_datetime("")

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current system date and time. If the user does not specify a format, use the default format '%Y-%m-%d %H:%M:%S'. Use this tool for requests involving the current date, time, timestamp, or formatted datetime values.",
  "strict": True,
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "Optional Python strftime format string. Default: '%Y-%m-%d %H:%M:%S'. Must not be empty."
      }
    },
    "additionalProperties": False
  }
})

## 5. Prepare a Tool-Enabled API Call
This cell builds a sample user message and sends a request scaffold with a tools list, which is the integration point where tool schemas are attached to Claude requests and multi-block responses begin.

[Open SS03 Lesson Notes: Tool Functions](../s06_tool_use_with_claude/ss03_tool_functions/index.md)</br>
[Open SS04 Lesson Notes: Tool Schemas](../s06_tool_use_with_claude/ss04_tool_schemas/index.md)</br>
[Open SS05 Lesson Notes: Handling Message Blocks](../s06_tool_use_with_claude/ss05_handling_message_blocks/index.md)</br>

In [49]:
messages = []

messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS"
})


response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
    tool_choice={"type": "tool", "name": "get_current_datetime"},
)


# print(response)

messages.append({
    "role": "assistant",
    "content": response.content
})

print(messages)




[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01AqLyFzBirdZ6detCKbvmZz', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]


In [50]:
tool_use_block = next((block for block in response.content if block.type == "tool_use"), None)

if tool_use_block is None:
    raise ValueError(f"No tool_use block returned. Blocks: {[block.type for block in response.content]}")

tool_args = tool_use_block.input
result = get_current_datetime(**tool_args)

## 6. Send the Tool Result Back
After extracting tool arguments and running the function, package the output as a `tool_result` block and append it as a user message.

[Open SS06 Lesson Notes: Sending Tool Results](../s06_tool_use_with_claude/ss06_sending_tool_results/index.md)

In [51]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_use_block.id,
        "content": result,
        "is_error": False
    }]
})

## 7. Verify Conversation State and Request Final Answer
Inspect the full conversation history, then make a follow-up call so Claude can read the tool result and produce the final natural-language response.

[Open SS06 Lesson Notes: Sending Tool Results](../s06_tool_use_with_claude/ss06_sending_tool_results/index.md)

In [52]:
# We now have the complete list of messages 
messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01AqLyFzBirdZ6detCKbvmZz', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01AqLyFzBirdZ6detCKbvmZz',
    'content': '12:14:03',
    'is_error': False}]}]

## 8. Send the Follow-up Request
Now call Claude again with the complete conversation (including the `tool_result` message). Keep the tool schema in the request so Claude can interpret prior tool references and return the final answer.

[Open SS06 Lesson Notes: Sending Tool Results](../s06_tool_use_with_claude/ss06_sending_tool_results/index.md)

In [53]:
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

final_response.content[0].text

'The exact time is **12:14:03**.'

## 6. Notebook Summary
You now have function implementations, JSON schemas, and a request pattern that can be extended with concrete tool registration and tool-result handling in later sections.

[Open SS03 Lesson Notes: Tool Functions](../s06_tool_use_with_claude/ss03_tool_functions/index.md)</br>
[Open SS04 Lesson Notes: Tool Schemas](../s06_tool_use_with_claude/ss04_tool_schemas/index.md)</br>
[Open SS05 Lesson Notes: Handling Message Blocks](../s06_tool_use_with_claude/ss05_handling_message_blocks/index.md)</br>
[Open SS06 Lesson Notes: Sending Tool Results](../s06_tool_use_with_claude/ss06_sending_tool_results/index.md)</br>